In [1]:
%matplotlib notebook
import sys, os
sys.path.append(os.path.abspath("../"))
import sam

In [1]:
import numpy as np
import matplotlib.pyplot as mpl
from poissolve.constants import nm, cm
from poissolve.solvers.poisson import PoissonSolver
from poissolve.solvers.fermi_dirac import FermiDirac3D
from poissolve.mesh import Mesh, EpiStack
from poissolve.mesh_functions import PointFunction,MidFunction,MaterialFunction,RegionFunction

ImportError: No module named 'poissolve'

In [ ]:
def plot_QFV(mesh):
    mpl.figure()
    mpl.subplot(311)
    mesh.plot_function('rho')
    mesh.plot_function('arho2')
    #mpl.ylim(-.00005,.00005)
    mpl.subplot(312)
    mesh.plot_function('E')
    mpl.subplot(313)
    mesh.plot_function('Ec','.')
    mesh.plot_function('Ev')
    mesh.plot_function('EF')
#plot_QFV(pn._mesh)

In [ ]:
class PN(): # SUPER
    def __init__(self,xp,xn,Nd,Na):
        epistack=EpiStack(['pGaN','GaN',xp],['nGaN','GaN',xn])
        self._mesh=Mesh(epistack,max_dz=1,refinements=[[xp,.1,1.3]])
        
        self._mesh.add_function('rho_p',PointFunction(self._mesh,arr=0.0))
        
        Nd=self._mesh.add_function('SiActiveConc',RegionFunction(self._mesh,
            lambda name: (name=="nGaN")*Nd, pos='point'))
        self._mesh.add_function('SiIonizedConc',PointFunction(self._mesh))
        
        # SPIKE IT! :D
        iint=self._mesh.interfaces_point[0][0]
        Nd.array[iint]=1e13*cm**-2/self._mesh._dzp[iint]
        
        
        self._mesh.add_function('MgActiveConc',RegionFunction(self._mesh,
            lambda name: (name=="pGaN")*Na, pos='point'))
        self._mesh.add_function('MgIonizedConc',PointFunction(self._mesh))
        
        self._mesh.add_function('EF',PointFunction(self._mesh,arr=0.0))
        self._mesh.add_function('rho',PointFunction(self._mesh,arr=0.0))
        self._arho2=self._mesh.add_function('arho2',PointFunction(self._mesh,arr=0.0))
        self._ps=PoissonSolver(self._mesh)
        self._fd=FermiDirac3D(self._mesh)
        
    def solve(self):
        self._ps.solve()
        for activation in np.logspace(-4,-0.,500):
            self._fd.solve(activation=activation)
            err=self._ps.isolve(visual=False)
        max_iter=100
        tol=1e-3
        i=0
        for i in range(max_iter):
            self._fd.solve(activation=1)
            err=self._ps.isolve(visual=False)
            if err<tol:
                print("Success")
                break
        assert err<tol, "Stopped because reached max_iter"
phib=3
#mpl.figure()
pn=PN(350*nm,550*nm,1.0e18*cm**-3,1.0e18*cm**-3)
pn.solve()


plot_QFV(pn._mesh)
#sam.kill_mplnb()

In [ ]:
len(pn._mesh.z)

In [ ]:
class samarr(np.ndarray):
    def __init__(name,arr=None):